# Phase 0-1 · 0-6: hold-out Val 표 · paired σ_diff · MDE

**목적:** `09_보완_재실험/B_결과/00_runs_raw.csv`(39런)에서 Phase 0용 표를 고정한다.

- **0-1** NCT hold-out Val 정확도 (tag × label_frac × seed)
- **0-6** lf=1.0 paired CRC 차분·σ_diff·n=3 MDE

**추가 학습·Kaggle 불필요.** 로컬 Run All.

**산출:** `03_val_sigma_mde/B_결과/01_holdout_val_table.md`, `02_paired_diff_mde.md`, `03_val_sigma_summary.json`

## 0. 경로

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
SEEDS = (42, 43, 44)
LF_ORDER = (0.1, 0.25, 1.0)
ALPHA = 0.05
POWER = 0.80


def find_phase0_root() -> Path:
    for c in [CWD, CWD.parent, CWD.parent.parent]:
        if (c / "03_val_sigma_mde").is_dir() or c.name.startswith("B_Phase0"):
            return c if c.name.startswith("B_Phase0") else c
        if (c / "01_split_overlap").is_dir():
            return c
    return CWD


def find_exp_root(phase0: Path) -> Path:
    local = phase0.parent.parent.parent / "09_보완_재실험"
    if (local / "B_결과" / "00_runs_raw.csv").exists():
        return local
    for c in [CWD, *CWD.parents]:
        if (c / "B_결과" / "00_runs_raw.csv").exists():
            return c
    raise FileNotFoundError("00_runs_raw.csv not found under 09_보완_재실험/B_결과/")


PHASE0_ROOT = find_phase0_root()
STEP3_ROOT = PHASE0_ROOT / "03_val_sigma_mde"
EXP_ROOT = find_exp_root(PHASE0_ROOT)
CSV_PATH = EXP_ROOT / "B_결과" / "00_runs_raw.csv"
DIR_RESULT = STEP3_ROOT / "B_결과"
DIR_RESULT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print("PHASE0_ROOT:", PHASE0_ROOT)
print("CSV:", CSV_PATH)
print("runs:", len(df))

## 1. hold-out Val 표 (0-1)

In [ ]:
def pivot_holdout_val(data: pd.DataFrame, lf: float) -> pd.DataFrame:
    sub = data[data["label_frac"] == lf].copy()
    rows = []
    for tag, g in sub.groupby("tag"):
        by_seed = {int(r.seed): r.best_holdout_acc for r in g.itertuples()}
        vals = [by_seed.get(s, float("nan")) for s in SEEDS]
        mean_v = float(np.nanmean(vals))
        std_v = float(np.nanstd(vals, ddof=0))
        rows.append({"tag": tag, "42": vals[0], "43": vals[1], "44": vals[2],
                       "mean": mean_v, "std_pop": std_v, "n": int(g["seed"].nunique())})
    out = pd.DataFrame(rows).sort_values("mean", ascending=False)
    return out


def md_holdout_table(data: pd.DataFrame, lf: float) -> str:
    p = pivot_holdout_val(data, lf)
    lines = [
        f"### label_frac = {lf}",
        "",
        "| tag | seed 42 | seed 43 | seed 44 | mean | std (pop) |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for r in p.itertuples():
        lines.append(
            f"| {r.tag} | {r._2:.4f} | {r._3:.4f} | {r._4:.4f} | {r.mean:.4f} | {r.std_pop:.4f} |"
        )
    return "\n".join(lines)


holdout_parts = [
    "# NCT hold-out Val 정확도 (best checkpoint 기준)",
    "",
    f"**출처:** `{CSV_PATH}`",
    f"**runs:** {len(df)} · monitor=`holdout_val` · std는 population (ddof=0, 메인 표와 동일)",
    "",
    "Val이 높고 CRC-VAL이 낮으면 내부(NCT) vs 외부(CRC) 괴리. Val≈0.97+ 이면 patch split 누수 가능성과 함께 해석.",
    "",
]
for lf in LF_ORDER:
    if lf in df["label_frac"].values:
        holdout_parts.append(md_holdout_table(df, lf))
        holdout_parts.append("")

# Val vs CRC scatter note for lf=1.0
lf1 = df[df["label_frac"] == 1.0].copy()
if not lf1.empty:
    holdout_parts.extend([
        "### lf=1.0 — Val vs CRC-VAL (seed별)",
        "",
        "| tag | seed | holdout Val | CRC Acc | Val−CRC |",
        "|---|---:|---:|---:|---:|",
    ])
    for r in lf1.sort_values(["tag", "seed"]).itertuples():
        holdout_parts.append(
            f"| {r.tag} | {int(r.seed)} | {r.best_holdout_acc:.4f} | {r.crc_acc_best:.4f} "
            f"| {r.best_holdout_acc - r.crc_acc_best:.4f} |"
        )
    holdout_parts.append("")

holdout_md = "\n".join(holdout_parts)
holdout_path = DIR_RESULT / "01_holdout_val_table.md"
holdout_path.write_text(holdout_md, encoding="utf-8")
print("Wrote:", holdout_path)

## 2. paired σ_diff · MDE (0-6)

In [ ]:
PAIRED_PAIRS = [
    ("curriculum", "none", "H3 proxy: curriculum vs plain SSL"),
    ("curriculum", "schedule_colorless", "H3: curriculum vs colorless schedule"),
    ("macenko_none", "curriculum", "H5: macenko vs curriculum"),
    ("curriculum", "imagenet_init", "H6 proxy: curriculum vs ImageNet"),
    ("schedule_colorless", "imagenet_init", "colorless vs ImageNet"),
]


def paired_diff_row(data: pd.DataFrame, tag_a: str, tag_b: str, lf: float) -> Dict:
    diffs = []
    seed_vals = {}
    for s in SEEDS:
        ra = data[(data.tag == tag_a) & (data.seed == s) & (data.label_frac == lf)]
        rb = data[(data.tag == tag_b) & (data.seed == s) & (data.label_frac == lf)]
        if len(ra) != 1 or len(rb) != 1:
            continue
        d = float(ra.iloc[0].crc_acc_best - rb.iloc[0].crc_acc_best)
        diffs.append(d)
        seed_vals[s] = d
    arr = np.array(diffs, dtype=float)
    return {
        "tag_a": tag_a,
        "tag_b": tag_b,
        "label_frac": lf,
        "diff_42": seed_vals.get(42, np.nan),
        "diff_43": seed_vals.get(43, np.nan),
        "diff_44": seed_vals.get(44, np.nan),
        "mean_diff": float(arr.mean()) if len(arr) else np.nan,
        "sigma_diff_pop": float(arr.std(ddof=0)) if len(arr) else np.nan,
        "sigma_diff_sample": float(arr.std(ddof=1)) if len(arr) > 1 else np.nan,
        "n_seed": len(arr),
    }


def mde_paired(sigma: float, n: int, alpha: float = ALPHA, power: float = POWER) -> float:
    """Minimum detectable paired mean difference (absolute, same units as Acc)."""
    if n < 2 or sigma <= 0 or math.isnan(sigma):
        return float("nan")
    try:
        from scipy.stats import t
        df = n - 1
        t_alpha = t.ppf(1 - alpha / 2, df)
        t_beta = t.ppf(power, df)
    except Exception:
        t_alpha, t_beta = 2.776, 0.886  # n=3 approx
    return float((t_alpha + t_beta) * sigma / math.sqrt(n))


paired_rows = []
for tag_a, tag_b, _note in PAIRED_PAIRS:
    row = paired_diff_row(df, tag_a, tag_b, 1.0)
    row["note"] = _note
    row["mde_80_pop"] = mde_paired(row["sigma_diff_pop"], int(row["n_seed"]))
    row["mde_80_sample"] = mde_paired(row["sigma_diff_sample"], int(row["n_seed"]))
    paired_rows.append(row)

paired_df = pd.DataFrame(paired_rows)

# curriculum lf=0.1 vs 1.0 (same seed)
lf_cmp = []
for s in SEEDS:
    a = df[(df.tag == "curriculum") & (df.seed == s) & (df.label_frac == 0.1)]
    b = df[(df.tag == "curriculum") & (df.seed == s) & (df.label_frac == 1.0)]
    if len(a) == 1 and len(b) == 1:
        lf_cmp.append({
            "seed": s,
            "crc_lf01": float(a.iloc[0].crc_acc_best),
            "crc_lf10": float(b.iloc[0].crc_acc_best),
            "delta": float(b.iloc[0].crc_acc_best - a.iloc[0].crc_acc_best),
        })
lf_cmp_df = pd.DataFrame(lf_cmp)

md_lines = [
    "# paired σ_diff · MDE (CRC-VAL Acc, lf=1.0)",
    "",
    f"**출처:** `{CSV_PATH}` · seed {list(SEEDS)} · **대응(paired)** 차분 (동일 split)",
    "",
    "## paired 차분 (A − B)",
    "",
    "| A − B | seed 42 | seed 43 | seed 44 | mean Δ | σ_diff (pop) | σ_diff (sample) | MDE 80% (pop σ) |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for r in paired_df.itertuples():
    label = f"{r.tag_a} − {r.tag_b}"
    md_lines.append(
        f"| {label} | {r.diff_42:+.4f} | {r.diff_43:+.4f} | {r.diff_44:+.4f} "
        f"| {r.mean_diff:+.4f} | {r.sigma_diff_pop:.4f} | {r.sigma_diff_sample:.4f} "
        f"| {r.mde_80_pop:.1%} |"
    )

md_lines.extend([
    "",
    "**MDE 해석:** n=3 paired, α=0.05, power=80% 기준. σ_diff≈0.05이면 약 **15–18 %p** 미만 차이는 검출 불가.",
    "H3 실제 gap(curriculum−colorless) ≈ 2.6 %p, H6 ≈ 1.4 %p → 현 설계로는 ‘Not supported’가 아니라 **검정력 부족**.",
    "",
    "### Discussion 초안 문장",
    "",
    "> With three paired seeds and σ_diff ≈ 0.05 on CRC-VAL accuracy, this design has ~80% power only for paired differences on the order of 15 percentage points or larger; smaller gaps (e.g., 1–3 pp between scheduled arms) cannot be ruled in or out.",
    "",
    "## curriculum: lf=0.1 vs lf=1.0 (CRC, seed별)",
    "",
    "| seed | CRC lf=0.1 | CRC lf=1.0 | Δ(1.0−0.1) |",
    "|---:|---:|---:|---:|",
])
for r in lf_cmp_df.itertuples():
    md_lines.append(f"| {int(r.seed)} | {r.crc_lf01:.4f} | {r.crc_lf10:.4f} | {r.delta:+.4f} |")
if not lf_cmp_df.empty:
    md_lines.append(
        f"\n**mean CRC:** lf=0.1 → {lf_cmp_df['crc_lf01'].mean():.4f}, "
        f"lf=1.0 → {lf_cmp_df['crc_lf10'].mean():.4f} (우연히 동일 mean 가능; seed별 Δ는 다름)."
    )

paired_md = "\n".join(md_lines) + "\n"
paired_path = DIR_RESULT / "02_paired_diff_mde.md"
paired_path.write_text(paired_md, encoding="utf-8")
print("Wrote:", paired_path)
print(paired_df[["tag_a", "tag_b", "mean_diff", "sigma_diff_pop", "mde_80_pop"]])

## 3. JSON 요약 저장

In [ ]:
summary = {
    "source_csv": str(CSV_PATH),
    "n_runs": int(len(df)),
    "holdout_val_lf1_tags": pivot_holdout_val(df, 1.0)[["tag", "mean", "std_pop"]].to_dict(orient="records"),
    "paired_lf1": paired_df.to_dict(orient="records"),
    "curriculum_lf_compare": lf_cmp_df.to_dict(orient="records"),
}
summary_path = DIR_RESULT / "03_val_sigma_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
paired_df.to_csv(DIR_RESULT / "04_paired_diff_lf1.csv", index=False)
print("Wrote:", summary_path)